In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import networkx as nx

from IPython.display import display

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed" / "string_v12_0"
METADATA = ROOT / "data" / "metadata"

OUT04 = ROOT / "outputs" / "notebook_04"
OUT05 = ROOT / "outputs" / "notebook_05"

OUT05.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Load frozen experiment design
# --------------------------------------------------

design = json.loads(
    (METADATA / "string_analysis_design_v12_0.json").read_text()
)

HLJ1_ID = design["hlj1_string_id"]
THRESHOLD = design["primary_score_threshold"]

assert THRESHOLD == 700

# --------------------------------------------------
# 3. Load graph-theoretic features
# --------------------------------------------------

baseline = pd.read_csv(
    ROOT / "outputs" / "notebook_02"
    / "protein_baseline_features_functional_700.csv"
)

assert baseline["string_protein_id"].is_unique

baseline = baseline.set_index("string_protein_id")

# --------------------------------------------------
# 4. Load Node2Vec consensus results
# --------------------------------------------------

n2v = pd.read_csv(OUT04 / "node2vec_consensus_unbiased.csv")

assert n2v["string_id"].is_unique

n2v = n2v.set_index("string_id")

rank_cols = [
    "rank_3888",
    "rank_3889",
    "rank_3890",
    "mean_rank",
    "std_rank",
    "top100_frequency"
]

assert n2v[rank_cols].notna().all().all()

# --------------------------------------------------
# 5. Load essential proteins
# --------------------------------------------------

essential_df = pd.read_csv(
    ROOT / "data" / "raw" / "essential_proteins.csv",
    header=None,
    names=[
        "sgd_id",
        "orf",
        "organism",
        "gene_name",
        "description"
    ],
    dtype=str
)

essential_orfs = set(essential_df["orf"].dropna())

# --------------------------------------------------
# 6. Reconstruct the primary network
# --------------------------------------------------

edges = pd.read_csv(
    PROCESSED / "functional_edges_score_ge_400.csv.gz",
    usecols=["u", "v", "combined_score"]
)

selected = edges.loc[
    edges["combined_score"] >= THRESHOLD,
    ["u", "v", "combined_score"]
]

G = nx.Graph()

G.add_nodes_from(baseline.index)

G.add_edges_from(
    (u, v, {"confidence": float(score) / 1000.0})
    for u, v, score in selected.itertuples(
        index=False,
        name=None
    )
)

# --------------------------------------------------
# 7. Extract HLJ1 connected component
# --------------------------------------------------

component_nodes = nx.node_connected_component(G, HLJ1_ID)

G_hlj1 = G.subgraph(component_nodes).copy()

# --------------------------------------------------
# 8. Validate node universe
# --------------------------------------------------

assert G_hlj1.number_of_nodes() == 5716
assert G_hlj1.number_of_edges() == 104138

candidate_universe = set(G_hlj1.nodes()) - {HLJ1_ID}

assert set(n2v.index) == candidate_universe

# --------------------------------------------------
# 9. Merge features
# --------------------------------------------------

panel = baseline.loc[sorted(candidate_universe)].copy()

panel = panel.join(
    n2v[rank_cols],
    how="left",
    validate="one_to_one"
)

assert panel["mean_rank"].notna().all()

panel["is_listed_essential"] = panel["orf"].isin(essential_orfs)

print("Notebook 05 inputs validated!")
print(f"Network nodes: {G_hlj1.number_of_nodes():,}")
print(f"Candidate universe: {len(panel):,}")
print(f"HLJ1 community: {baseline.at[HLJ1_ID, 'community_id']}")

Notebook 05 inputs validated!
Network nodes: 5,716
Candidate universe: 5,715
HLJ1 community: 4


In [3]:
# --------------------------------------------------
# 1. HLJ1-personalized PageRank
# --------------------------------------------------

rwr_scores = nx.pagerank(
    G_hlj1,
    alpha=0.85,
    personalization={HLJ1_ID: 1.0},
    weight="confidence",
    tol=1e-9
)

panel["rwr_score"] = pd.Series(rwr_scores).reindex(panel.index)

# --------------------------------------------------
# 2. HLJ1 community
# --------------------------------------------------

hlj1_community = baseline.at[HLJ1_ID, "community_id"]

community_nodes = {
    v for v in G_hlj1.nodes()
    if baseline.at[v, "community_id"] == hlj1_community
}

# Weighted volume of the full network
total_volume = sum(
    strength
    for _, strength in G_hlj1.degree(
        weight="confidence"
    )
)

# Weighted volume of HLJ1's community
community_volume = sum(
    G_hlj1.degree(v, weight="confidence")
    for v in community_nodes
)

community_prior = community_volume / total_volume

# --------------------------------------------------
# 3. Smoothed community affinity
# --------------------------------------------------

LAMBDA = 5.0

def community_affinity(v):

    # Total weighted degree
    strength = G_hlj1.degree(v, weight="confidence")

    # Weighted edges leading into HLJ1 community
    affinity = sum(
        data["confidence"]
        for _, neighbor, data in G_hlj1.edges(
            v, data=True
        )
        if neighbor in community_nodes
    )

    return (
        affinity + LAMBDA * community_prior
    ) / (strength + LAMBDA)


panel["community_affinity"] = [
    community_affinity(v)
    for v in panel.index
]

# --------------------------------------------------
# 4. Convert to percentile scores
# --------------------------------------------------

def percentile_score(values):
    """
    Convert scores to [0,1] rank percentiles.

    Higher original values receive higher scores.
    Ties receive average ranks.
    """

    ranks = values.rank(ascending=False, method="average")

    n = len(values)

    return 1.0 - (ranks - 1.0) / (n - 1.0)


# Smaller mean rank is better
panel["N"] = percentile_score(-panel["mean_rank"])

# Higher RWR probability is better
panel["R"] = percentile_score(panel["rwr_score"])

# Higher community affinity is better
panel["C"] = percentile_score(panel["community_affinity"])

# Validate component scores
assert (
    panel[["N", "R", "C"]]
    .apply(lambda col: col.between(0, 1).all())
    .all()
)

display(panel[["N", "R", "C"]].describe())

,N,R,C
count,5715.000000,5715.000000,5715.000000
mean,0.500000,0.500000,0.500000
std,0.288751,0.288751,0.288751
min,0.000000,0.000000,0.000000
25%,0.250000,0.250000,0.249956
50%,0.500000,0.500000,0.500088
75%,0.750000,0.750000,0.749869
max,1.000000,1.000000,1.000000


In [4]:
# --------------------------------------------------
# 1. Equal-weight hybrid baseline
# --------------------------------------------------

WEIGHTS = {
    "N": 1 / 3,
    "R": 1 / 3,
    "C": 1 / 3,
}

panel["hybrid_score"] = sum(
    WEIGHTS[col] * panel[col]
    for col in WEIGHTS
)

panel["hybrid_rank"] = (
    panel["hybrid_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# --------------------------------------------------
# 2. Define provisional experimental eligibility
# --------------------------------------------------

# Exclude:
# - Predefined reference proteins
# - Unmapped ORFs
# - Proteins listed as essential

panel["provisional_eligible"] = (
    panel["candidate_eligible"].eq(True)
    & panel["orf"].notna()
    & ~panel["is_listed_essential"]
)

# --------------------------------------------------
# 3. Generate candidate shortlist
# --------------------------------------------------

hybrid_candidates = (
    panel.loc[panel["provisional_eligible"]]
    .sort_values("hybrid_score", ascending=False)
    .copy()
)

columns = [
    "orf",
    "preferred_name",
    "hybrid_rank",
    "hybrid_score",
    "N",
    "R",
    "C",
    "mean_rank",
    "std_rank",
    "degree",
    "h_hops",
    "community_id",
    "community_affinity",
]

print("Top-20 Hybrid Candidates:")

display(hybrid_candidates[columns].head(20))

# --------------------------------------------------
# 4. Save results
# --------------------------------------------------

panel.reset_index().to_csv(
    OUT05 / "hybrid_all_proteins.csv",
    index=False
)

hybrid_candidates.reset_index().to_csv(
    OUT05 / "hybrid_candidate_ranking.csv",
    index=False
)

print("Hybrid ranking completed and saved!")

Top-20 Hybrid Candidates:


,orf,preferred_name,hybrid_rank,hybrid_score,N,R,C,mean_rank,std_rank,degree,h_hops,community_id,community_affinity
string_protein_id,,,,,,,,,,,,,
4932.YPL106C,YPL106C,SSE1,2,0.998862,0.998687,0.998775,0.999125,9.666667,4.163332,51,1.0,4,0.795183
4932.YJL162C,YJL162C,JJJ2,3,0.997667,1.000000,0.994575,0.998425,1.000000,0.000000,31,2.0,4,0.782782
4932.YJR097W,YJR097W,JJJ3,4,0.997550,0.996675,0.998425,0.997550,20.666667,2.081666,32,1.0,4,0.772226
4932.YBL075C,YBL075C,SSA3,5,0.997462,0.997287,0.997375,0.997725,15.000000,7.000000,32,1.0,4,0.776001
4932.YNL077W,YNL077W,APJ1,6,0.997258,0.999825,0.995275,0.996675,3.666667,2.886751,38,2.0,4,0.763352
4932.YER103W,YER103W,SSA4,7,0.997171,0.998162,0.997200,0.996150,10.666667,9.865766,36,1.0,4,0.756546
4932.YBR169C,YBR169C,SSE2,8,0.997112,0.997637,0.994050,0.999650,13.333333,4.163332,42,2.0,4,0.818798
4932.YKL073W,YKL073W,LHS1,9,0.996733,0.996850,0.996500,0.996850,20.000000,3.605551,32,1.0,4,0.763742
4932.YMR214W,YMR214W,SCJ1,10,0.996500,0.996150,0.994750,0.998600,22.666667,4.163332,40,2.0,4,0.784665


Hybrid ranking completed and saved!


In [5]:
from itertools import combinations

ABLATION_MODELS = {
    "N": ["N"],
    "R": ["R"],
    "C": ["C"],
    "N+R": ["N", "R"],
    "N+C": ["N", "C"],
    "R+C": ["R", "C"],
    "Full": ["N", "R", "C"],
}

# Generate scores and ranks
for name, components in ABLATION_MODELS.items():

    panel[f"score_{name}"] = (
        panel[components].mean(axis=1)
    )

    panel[f"rank_{name}"] = (
        panel[f"score_{name}"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

print("Ablation rankings computed.")

Ablation rankings computed.


In [6]:
K_VALUES = [20, 50, 100, 200]

eligible = panel[
    panel["provisional_eligible"]
].copy()

records = []

for model_name in ABLATION_MODELS:

    for k in K_VALUES:

        topk = eligible.nlargest(
            k,
            f"score_{model_name}"
        )

        community_fraction = (
            topk["same_community_as_HLJ1"]
            .mean()
        )

        records.append({
            "model": model_name,
            "top_k": k,
            "same_hlj1_community_fraction":
                community_fraction,
            "other_community_count":
                int((~topk["same_community_as_HLJ1"]).sum()),
        })

community_bias_df = pd.DataFrame(records)

display(
    community_bias_df.pivot(
        index="model",
        columns="top_k",
        values="same_hlj1_community_fraction"
    ).round(3)
)

top_k,20,50,100,200
model,,,,
C,1.0,1.00,1.00,1.000
Full,1.0,1.00,0.81,0.610
N,1.0,0.96,0.76,0.505
N+C,1.0,1.00,0.97,0.745
N+R,1.0,0.86,0.70,0.425
R,0.9,0.80,0.54,0.320
R+C,1.0,1.00,0.81,0.585


In [7]:
# Five predefined ERAD seeds + KAR2 + YDJ1
reference_orfs = set(design["core_seed_orfs"]) | {"YJL034W", "YNL064C"}

reference_orfs.discard("YMR161W")

rank_columns = [f"rank_{name}" for name in ABLATION_MODELS]

reference_ablation = panel[panel["orf"].isin(reference_orfs)][
    ["orf", "preferred_name", "community_id"] + rank_columns
].copy()

reference_ablation = reference_ablation.sort_values("rank_Full")

display(reference_ablation)

,orf,preferred_name,community_id,rank_N,rank_R,rank_C,rank_N+R,rank_N+C,rank_R+C,rank_Full
string_protein_id,,,,,,,,,,
4932.YNL064C,YNL064C,YDJ1,4,37,24,71,31,23,20,22
4932.YJL034W,YJL034W,KAR2,4,62,6,149,34,45,30,34
4932.YOL013C,YOL013C,HRD1,9,124,118,493,88,167,110,105
4932.YBR201W,YBR201W,DER1,9,110,176,501,98,164,117,108
4932.YLR207W,YLR207W,HRD3,9,102,182,580,97,184,130,116
4932.YMR022W,YMR022W,UBC7,9,144,286,567,132,193,153,134
4932.YMR264W,YMR264W,CUE1,9,142,251,984,122,333,243,185


In [8]:
# --------------------------------------------------
# 1. Define two scoring functions
# --------------------------------------------------

panel["H_local"] = panel[["N", "R", "C"]].mean(axis=1)

panel["H_cross"] = panel[["N", "R"]].mean(axis=1)

# --------------------------------------------------
# 2. Assign proteins to tracks
# --------------------------------------------------

panel["track"] = np.where(
    panel["community_id"] == hlj1_community,
    "A_local",
    "B_cross"
)

# --------------------------------------------------
# 3. Compute track-specific rankings
# --------------------------------------------------

panel["track_score"] = np.where(
    panel["track"] == "A_local",
    panel["H_local"],
    panel["H_cross"]
)

# Ranks within each track, before biological filtering
panel["track_rank_all"] = (
    panel.groupby("track")["track_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# --------------------------------------------------
# 4. Apply provisional candidate eligibility
# --------------------------------------------------

eligible = panel.loc[
    panel["provisional_eligible"]
].copy()

# Rank eligible proteins within each track
eligible["eligible_track_rank"] = (
    eligible.groupby("track")["track_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# --------------------------------------------------
# 5. Extract separate rankings
# --------------------------------------------------

local_candidates = (
    eligible[eligible["track"] == "A_local"]
    .sort_values("track_score", ascending=False)
    .copy()
)

cross_candidates = (
    eligible[eligible["track"] == "B_cross"]
    .sort_values("track_score", ascending=False)
    .copy()
)

# --------------------------------------------------
# 6. Validate
# --------------------------------------------------

assert set(local_candidates.index).isdisjoint(
    set(cross_candidates.index)
)

assert len(local_candidates) + len(cross_candidates) == len(eligible)

print("Dual-track algorithm completed!")

print(f"Local candidates: {len(local_candidates):,}")
print(f"Cross-community candidates: {len(cross_candidates):,}")

Dual-track algorithm completed!
Local candidates: 340
Cross-community candidates: 4,099


In [9]:
# Add distance from HLJ1
distances = nx.single_source_shortest_path_length(
    G_hlj1,
    HLJ1_ID
)

eligible["distance"] = eligible.index.map(distances)

local_candidates = eligible[
    eligible["track"] == "A_local"
].sort_values("track_score", ascending=False)

cross_candidates = eligible[
    eligible["track"] == "B_cross"
].sort_values("track_score", ascending=False)

columns = [
    "orf",
    "preferred_name",
    "community_id",
    "eligible_track_rank",
    "track_score",
    "N",
    "R",
    "C",
    "distance",
    "degree",
]

print("TRACK A — Local Proteostasis Candidates")
display(local_candidates[columns].head(15))

print("\nTRACK B — Cross-Community Candidates")
display(cross_candidates[columns].head(15))

# Save
local_candidates.reset_index().to_csv(
    OUT05 / "track_A_local_candidates.csv",
    index=False
)

cross_candidates.reset_index().to_csv(
    OUT05 / "track_B_cross_community_candidates.csv",
    index=False
)

TRACK A — Local Proteostasis Candidates


,orf,preferred_name,community_id,eligible_track_rank,track_score,N,R,C,distance,degree
string_protein_id,,,,,,,,,,
4932.YPL106C,YPL106C,SSE1,4,1,0.998862,0.998687,0.998775,0.999125,1,51
4932.YJL162C,YJL162C,JJJ2,4,2,0.997667,1.000000,0.994575,0.998425,2,31
4932.YJR097W,YJR097W,JJJ3,4,3,0.997550,0.996675,0.998425,0.997550,1,32
4932.YBL075C,YBL075C,SSA3,4,4,0.997462,0.997287,0.997375,0.997725,1,32
4932.YNL077W,YNL077W,APJ1,4,5,0.997258,0.999825,0.995275,0.996675,2,38
4932.YER103W,YER103W,SSA4,4,6,0.997171,0.998162,0.997200,0.996150,1,36
4932.YBR169C,YBR169C,SSE2,4,7,0.997112,0.997637,0.994050,0.999650,2,42
4932.YKL073W,YKL073W,LHS1,4,8,0.996733,0.996850,0.996500,0.996850,1,32
4932.YMR214W,YMR214W,SCJ1,4,9,0.996500,0.996150,0.994750,0.998600,2,40



TRACK B — Cross-Community Candidates


,orf,preferred_name,community_id,eligible_track_rank,track_score,N,R,C,distance,degree
string_protein_id,,,,,,,,,,
4932.YIL030C,YIL030C,SSM4,9,1,0.991162,0.986699,0.995625,0.875044,1,34
4932.YDL190C,YDL190C,UFD2,9,2,0.990200,0.984949,0.995450,0.823941,1,28
4932.YBR101C,YBR101C,FES1,5,3,0.988712,0.984074,0.993350,0.909695,2,238
4932.YGR033C,YGR033C,TIM21,7,4,0.987224,0.986699,0.987749,0.868743,2,33
4932.YKR065C,YKR065C,PAM17,7,5,0.986043,0.988012,0.984074,0.897270,2,17
4932.YPL098C,YPL098C,MGR2,7,6,0.985256,0.988012,0.982499,0.884319,2,20
4932.YNL121C,YNL121C,TOM70,7,7,0.984599,0.979174,0.990025,0.870669,2,59
4932.YJR135W-A,YJR135W-A,TIM8,7,8,0.980399,0.983024,0.977774,0.852643,2,32
4932.YHR117W,YHR117W,TOM71,7,9,0.980355,0.990287,0.970424,0.933672,2,13


In [10]:
# --------------------------------------------------
# 1. Same evaluation universe for both scores
# --------------------------------------------------

cross_pool = panel[panel["track"] == "B_cross"].copy()

# --------------------------------------------------
# 2. Rank using different scoring functions
# --------------------------------------------------

METHODS = {
    "Node2Vec": "N",
    "RWR": "R",
    "N+R": "H_cross",
    "Full": "H_local",
}

for name, col in METHODS.items():

    cross_pool[f"rank_{name}"] = (
        cross_pool[col]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

# --------------------------------------------------
# 3. Inspect predefined ERAD references
# --------------------------------------------------

erad_reference_orfs = [
    orf for orf in design["core_seed_orfs"]
    if orf != "YMR161W"
]

reference_comparison = cross_pool[
    cross_pool["orf"].isin(erad_reference_orfs)
][[
    "orf",
    "preferred_name",
    "community_id",
    "rank_Node2Vec",
    "rank_RWR",
    "rank_N+R",
    "rank_Full",
]].copy()

reference_comparison["rank_improvement"] = (
    reference_comparison["rank_Full"]
    - reference_comparison["rank_N+R"]
)

display(
    reference_comparison.sort_values("rank_N+R")
)

,orf,preferred_name,community_id,rank_Node2Vec,rank_RWR,rank_N+R,rank_Full,rank_improvement
string_protein_id,,,,,,,,
4932.YOL013C,YOL013C,HRD1,9,35,52,25,7,-18
4932.YLR207W,YLR207W,HRD3,9,22,105,28,15,-13
4932.YBR201W,YBR201W,DER1,9,27,99,29,9,-20
4932.YMR264W,YMR264W,CUE1,9,44,164,44,57,13
4932.YMR022W,YMR022W,UBC7,9,46,195,47,25,-22


In [11]:
# --------------------------------------------------
# 1. Same candidate universe
# --------------------------------------------------

cross_pool = panel.loc[
    panel["track"] == "B_cross"
].copy()

cross_eligible = cross_pool.loc[
    cross_pool["provisional_eligible"]
].copy()

METHODS = {
    "N+R": "H_cross",
    "N+R+C": "H_local",
}

# --------------------------------------------------
# 2. Compare Top-k composition
# --------------------------------------------------

records = []

for method, score_col in METHODS.items():

    for k in [20, 50, 100]:

        topk = cross_eligible.nlargest(k, score_col)

        counts = topk["community_id"].value_counts()

        records.append({
            "method": method,
            "top_k": k,
            "community_9": counts.get(9, 0),
            "community_7": counts.get(7, 0),
            "community_5": counts.get(5, 0),
            "unique_communities":
                topk["community_id"].nunique()
        })

composition_df = pd.DataFrame(records)

display(composition_df)

# --------------------------------------------------
# 3. Overall ranking agreement
# --------------------------------------------------

rho = cross_eligible["H_cross"].corr(
    cross_eligible["H_local"],
    method="spearman"
)

print(f"Spearman correlation: {rho:.4f}")

# --------------------------------------------------
# 4. Top-k overlap
# --------------------------------------------------

overlap_records = []

for k in [20, 50, 100]:

    A = set(
        cross_eligible.nlargest(
            k, "H_cross"
        ).index
    )

    B = set(
        cross_eligible.nlargest(
            k, "H_local"
        ).index
    )

    overlap_records.append({
        "top_k": k,
        "intersection": len(A & B),
        "jaccard": len(A & B) / len(A | B)
    })

display(pd.DataFrame(overlap_records))

,method,top_k,community_9,community_7,community_5,unique_communities
0,N+R,20,8,8,3,4
1,N+R,50,13,11,12,7
2,N+R,100,16,15,36,10
3,N+R+C,20,8,6,3,5
4,N+R+C,50,11,8,5,8
5,N+R+C,100,14,19,6,11


Spearman correlation: 0.8704


,top_k,intersection,jaccard
0,20,14,0.538462
1,50,30,0.428571
2,100,45,0.290323


In [12]:
# --------------------------------------------------
# 1. Weighted association with HLJ1 community
# --------------------------------------------------

def weighted_links_to_hlj1_community(v):

    return sum(
        data["confidence"]
        for _, neighbor, data in G_hlj1.edges(
            v, data=True
        )
        if neighbor in community_nodes
    )


cross_pool["links_to_hlj1_community"] = [
    weighted_links_to_hlj1_community(v)
    for v in cross_pool.index
]

cross_pool["weighted_degree"] = [
    G_hlj1.degree(v, weight="confidence")
    for v in cross_pool.index
]

# --------------------------------------------------
# 2. Inspect selected proteins
# --------------------------------------------------

interesting_genes = [
    "SSM4",
    "UFD2",
    "UBX2",
    "NPL4",
    "SIL1",
    "HRD1",
    "DER1",
    "HRD3",
    "UBC7",
    "CUE1",
]

diagnostics = cross_pool[
    cross_pool["preferred_name"].isin(
        interesting_genes
    )
].copy()

display(
    diagnostics[
        [
            "orf",
            "preferred_name",
            "community_id",
            "weighted_degree",
            "links_to_hlj1_community",
            "community_affinity",
            "N",
            "R",
            "C",
            "H_cross",
            "H_local",
        ]
    ].sort_values(
        "community_affinity",
        ascending=False
    )
)

,orf,preferred_name,community_id,weighted_degree,links_to_hlj1_community,community_affinity,N,R,C,H_cross,H_local
string_protein_id,,,,,,,,,,,
4932.YOL013C,YOL013C,HRD1,9,32.057,5.025,0.139937,0.978299,0.979524,0.913896,0.978911,0.957240
4932.YBR201W,YBR201W,DER1,9,20.751,3.287,0.133884,0.980924,0.969373,0.912496,0.975149,0.954264
4932.YMR022W,YMR022W,UBC7,9,22.218,2.798,0.108702,0.974974,0.950123,0.900945,0.962548,0.942014
4932.YLR207W,YLR207W,HRD3,9,21.050,2.552,0.104132,0.982237,0.968323,0.898670,0.975280,0.949743
4932.YIL030C,YIL030C,SSM4,9,30.058,2.262,0.069104,0.986699,0.995625,0.875044,0.991162,0.952456
4932.YOL031C,YOL031C,SIL1,5,178.061,7.429,0.041460,0.971649,0.986874,0.840042,0.979261,0.932855
4932.YMR264W,YMR264W,CUE1,9,23.016,0.775,0.033397,0.975236,0.956248,0.827966,0.965742,0.919817
4932.YDL190C,YDL190C,UFD2,9,24.445,0.741,0.030621,0.984949,0.995450,0.823941,0.990200,0.934780
4932.YML013W,YML013W,UBX2,9,28.133,0.715,0.026428,0.977686,0.980574,0.768813,0.979130,0.909025


In [13]:
import numpy as np
import pandas as pd
import networkx as nx

# --------------------------------------------------
# 1. Settings
# --------------------------------------------------

LOUVAIN_SEEDS = list(range(3888, 3898))

# Original HLJ1 community from Notebook 02
original_hlj1_community = set(community_nodes)

# Store results
louvain_records = []
membership_records = []

# --------------------------------------------------
# 2. Repeated community detection
# --------------------------------------------------

for seed in LOUVAIN_SEEDS:

    print(f"Running Louvain: seed={seed}")

    communities = nx.community.louvain_communities(
        G,
        weight="confidence",
        resolution=1.0,
        seed=seed
    )

    # Find the community containing HLJ1
    hlj1_cluster = next(
        community
        for community in communities
        if HLJ1_ID in community
    )

    hlj1_cluster = set(hlj1_cluster)

    # Compare with original partition
    intersection = len(hlj1_cluster & original_hlj1_community)

    union = len(hlj1_cluster | original_hlj1_community)

    jaccard = intersection / union

    louvain_records.append({
        "seed": seed,
        "num_communities": len(communities),
        "hlj1_community_size": len(hlj1_cluster),
        "jaccard_with_original": jaccard
    })

    # Track membership for every node in HLJ1 component
    for protein in G_hlj1.nodes():

        membership_records.append({
            "seed": seed,
            "string_id": protein,
            "in_hlj1_community":
                protein in hlj1_cluster
        })

# --------------------------------------------------
# 3. Summary
# --------------------------------------------------

louvain_summary = pd.DataFrame(louvain_records)

membership_df = pd.DataFrame(membership_records)

display(louvain_summary)

print("\nMean HLJ1 community Jaccard:")
print(louvain_summary["jaccard_with_original"].mean())

print("\nHLJ1 community size:")
print(louvain_summary["hlj1_community_size"].describe())

Running Louvain: seed=3888
Running Louvain: seed=3889
Running Louvain: seed=3890
Running Louvain: seed=3891
Running Louvain: seed=3892
Running Louvain: seed=3893
Running Louvain: seed=3894
Running Louvain: seed=3895
Running Louvain: seed=3896
Running Louvain: seed=3897


,seed,num_communities,hlj1_community_size,jaccard_with_original
0,3888,860,435,1.000000
1,3889,861,650,0.273474
2,3890,863,513,0.350427
3,3891,860,442,0.827083
4,3892,859,322,0.493097
5,3893,862,720,0.211962
6,3894,862,508,0.671986
7,3895,862,317,0.606838
8,3896,863,413,0.420436
9,3897,861,298,0.536688



Mean HLJ1 community Jaccard:
0.5391990298016751

HLJ1 community size:
count     10.000000
mean     461.800000
std      140.505279
min      298.000000
25%      344.750000
50%      438.500000
75%      511.750000
max      720.000000
Name: hlj1_community_size, dtype: float64


In [14]:
# --------------------------------------------------
# 1. Compute membership frequency
# --------------------------------------------------

membership_frequency = (
    membership_df
    .groupby("string_id")["in_hlj1_community"]
    .mean()
)

# --------------------------------------------------
# 2. Select candidates for investigation
# --------------------------------------------------

tracked_ids = (
    set(local_candidates.head(15).index)
    | set(cross_candidates.head(15).index)
)

# Include biological references
reference_ids = {
    f"4932.{orf}"
    for orf in reference_orfs
}

tracked_ids |= reference_ids

# --------------------------------------------------
# 3. Build diagnostic table
# --------------------------------------------------

track_stability = panel.loc[
    sorted(tracked_ids),
    [
        "orf",
        "preferred_name",
        "community_id",
        "track",
        "N",
        "R",
        "C"
    ]
].copy()

track_stability["hlj1_membership_frequency"] = (
    track_stability.index.map(
        membership_frequency
    )
)

# Fraction of runs assigned to the other track
track_stability["switch_frequency"] = np.where(
    track_stability["track"] == "A_local",
    1 - track_stability["hlj1_membership_frequency"],
    track_stability["hlj1_membership_frequency"]
)

# --------------------------------------------------
# 4. Display
# --------------------------------------------------

display(
    track_stability[
        [
            "orf",
            "preferred_name",
            "track",
            "hlj1_membership_frequency",
            "switch_frequency"
        ]
    ].sort_values(
        "switch_frequency",
        ascending=False
    )
)

,orf,preferred_name,track,hlj1_membership_frequency,switch_frequency
string_protein_id,,,,,
4932.YJR135W-A,YJR135W-A,TIM8,B_cross,0.4,0.4
4932.YHR117W,YHR117W,TOM71,B_cross,0.4,0.4
4932.YKR065C,YKR065C,PAM17,B_cross,0.4,0.4
4932.YNL070W,YNL070W,TOM7,B_cross,0.4,0.4
4932.YPL098C,YPL098C,MGR2,B_cross,0.4,0.4
4932.YNL121C,YNL121C,TOM70,B_cross,0.4,0.4
4932.YGR033C,YGR033C,TIM21,B_cross,0.4,0.4
4932.YGR181W,YGR181W,TIM13,B_cross,0.4,0.4
4932.YLR207W,YLR207W,HRD3,B_cross,0.2,0.2


In [18]:
# --------------------------------------------------
# 1. Compute consensus membership
# --------------------------------------------------

membership_frequency = (
    membership_df
    .groupby("string_id")["in_hlj1_community"]
    .mean()
)

assert set(panel.index).issubset(
    set(membership_frequency.index)
)

panel["membership_frequency"] = (
    membership_frequency.reindex(panel.index)
)

# --------------------------------------------------
# 2. Consensus-based classification
# --------------------------------------------------

CONSENSUS_THRESHOLD = 0.8

panel["consensus_track"] = np.select(
    [
        panel["membership_frequency"] >= CONSENSUS_THRESHOLD,
        panel["membership_frequency"] <= 1 - CONSENSUS_THRESHOLD,
    ],
    [
        "A_local_consensus",
        "B_cross_consensus",
    ],
    default="Boundary"
)

# --------------------------------------------------
# 3. Summary
# --------------------------------------------------

print("Consensus track distribution:")

display(
    panel["consensus_track"].value_counts()
)

print("\nOriginal vs consensus track:")

display(
    pd.crosstab(
        panel["track"],
        panel["consensus_track"]
    )
)

# --------------------------------------------------
# 4. Save
# --------------------------------------------------

panel[
    [
        "orf",
        "preferred_name",
        "community_id",
        "track",
        "membership_frequency",
        "consensus_track"
    ]
].to_csv(
    OUT05 / "community_consensus_membership.csv"
)

print("Consensus membership saved.")

Consensus track distribution:


consensus_track
B_cross_consensus    4743
Boundary              805
A_local_consensus     167
Name: count, dtype: int64


Original vs consensus track:


consensus_track,A_local_consensus,B_cross_consensus,Boundary
track,,,
A_local,167,1,266
B_cross,0,4742,539


Consensus membership saved.


In [20]:
# --------------------------------------------------
# 1. Local Top-15
# --------------------------------------------------

local_stability = (
    local_candidates.head(15)[
        [
            "orf",
            "preferred_name",
            "eligible_track_rank",
        ]
    ]
    .join(
        panel[
            [
                "membership_frequency",
                "consensus_track",
            ]
        ],
        how="left"
    )
)

print("TRACK A — Candidate Stability")
display(local_stability)


# --------------------------------------------------
# 2. Cross-community Top-15
# --------------------------------------------------

cross_stability = (
    cross_candidates.head(15)[[
        "orf",
        "preferred_name",
        "eligible_track_rank",
    ]].join(panel[[
        "membership_frequency",
        "consensus_track",
    ]], how="left")
)

print("\nTRACK B — Candidate Stability")
display(cross_stability)


# --------------------------------------------------
# 3. Validation
# --------------------------------------------------

assert local_stability["membership_frequency"].notna().all()
assert cross_stability["membership_frequency"].notna().all()


# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

print("\nTrack A consensus classification:")
print(local_stability["consensus_track"].value_counts())

print("\nTrack B consensus classification:")
print(cross_stability["consensus_track"].value_counts())

TRACK A — Candidate Stability


,orf,preferred_name,eligible_track_rank,membership_frequency,consensus_track
string_protein_id,,,,,
4932.YPL106C,YPL106C,SSE1,1,1.0,A_local_consensus
4932.YJL162C,YJL162C,JJJ2,2,1.0,A_local_consensus
4932.YJR097W,YJR097W,JJJ3,3,1.0,A_local_consensus
4932.YBL075C,YBL075C,SSA3,4,1.0,A_local_consensus
4932.YNL077W,YNL077W,APJ1,5,1.0,A_local_consensus
4932.YER103W,YER103W,SSA4,6,1.0,A_local_consensus
4932.YBR169C,YBR169C,SSE2,7,1.0,A_local_consensus
4932.YKL073W,YKL073W,LHS1,8,1.0,A_local_consensus
4932.YMR214W,YMR214W,SCJ1,9,1.0,A_local_consensus



TRACK B — Candidate Stability


,orf,preferred_name,eligible_track_rank,membership_frequency,consensus_track
string_protein_id,,,,,
4932.YIL030C,YIL030C,SSM4,1,0.2,Boundary
4932.YDL190C,YDL190C,UFD2,2,0.2,Boundary
4932.YBR101C,YBR101C,FES1,3,0.0,B_cross_consensus
4932.YGR033C,YGR033C,TIM21,4,0.4,Boundary
4932.YKR065C,YKR065C,PAM17,5,0.4,Boundary
4932.YPL098C,YPL098C,MGR2,6,0.4,Boundary
4932.YNL121C,YNL121C,TOM70,7,0.4,Boundary
4932.YJR135W-A,YJR135W-A,TIM8,8,0.4,Boundary
4932.YHR117W,YHR117W,TOM71,9,0.4,Boundary



Track A consensus classification:
consensus_track
A_local_consensus    15
Name: count, dtype: int64

Track B consensus classification:
consensus_track
Boundary             12
B_cross_consensus     3
Name: count, dtype: int64


In [21]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Count Louvain membership votes
# --------------------------------------------------

membership_counts = (
    membership_df
    .groupby("string_id")["in_hlj1_community"]
    .agg(["sum", "count"])
)

membership_counts.columns = [
    "membership_count",
    "num_runs"
]

# Validate equal number of runs per protein
assert membership_counts["num_runs"].eq(10).all()

# --------------------------------------------------
# 2. Update panel
# --------------------------------------------------

panel["membership_count"] = (
    membership_counts["membership_count"]
    .reindex(panel.index)
    .astype(int)
)

panel["membership_frequency"] = (
    panel["membership_count"] / 10
)

# --------------------------------------------------
# 3. Integer-based classification
# --------------------------------------------------

panel["consensus_track"] = np.select(
    [
        panel["membership_count"] >= 8,
        panel["membership_count"] <= 2,
    ],
    [
        "A_local_consensus",
        "B_cross_consensus",
    ],
    default="Boundary"
)

# --------------------------------------------------
# 4. Validate
# --------------------------------------------------

assert panel["consensus_track"].notna().all()

assert (
    panel.loc[
        panel["membership_count"] == 2,
        "consensus_track"
    ] == "B_cross_consensus"
).all()

# --------------------------------------------------
# 5. Display results
# --------------------------------------------------

print("Corrected consensus distribution:")

display(
    panel["consensus_track"].value_counts()
)

print("\nOriginal vs corrected consensus:")

display(
    pd.crosstab(
        panel["track"],
        panel["consensus_track"]
    )
)

Corrected consensus distribution:


consensus_track
B_cross_consensus    5139
Boundary              409
A_local_consensus     167
Name: count, dtype: int64


Original vs corrected consensus:


consensus_track,A_local_consensus,B_cross_consensus,Boundary
track,,,
A_local,167,35,232
B_cross,0,5104,177


In [22]:
def candidate_stability_table(candidates, panel, k=15):

    result = (
        candidates.head(k)[
            [
                "orf",
                "preferred_name",
                "eligible_track_rank",
            ]
        ]
        .join(
            panel[
                [
                    "membership_count",
                    "membership_frequency",
                    "consensus_track",
                ]
            ],
            how="left"
        )
    )

    return result


local_stability = candidate_stability_table(
    local_candidates,
    panel
)

cross_stability = candidate_stability_table(
    cross_candidates,
    panel
)

print("TRACK A")
display(local_stability)

print("\nTRACK B")
display(cross_stability)

print("\nTrack A classification:")
print(local_stability["consensus_track"].value_counts())

print("\nTrack B classification:")
print(cross_stability["consensus_track"].value_counts())

TRACK A


,orf,preferred_name,eligible_track_rank,membership_count,membership_frequency,consensus_track
string_protein_id,,,,,,
4932.YPL106C,YPL106C,SSE1,1,10,1.0,A_local_consensus
4932.YJL162C,YJL162C,JJJ2,2,10,1.0,A_local_consensus
4932.YJR097W,YJR097W,JJJ3,3,10,1.0,A_local_consensus
4932.YBL075C,YBL075C,SSA3,4,10,1.0,A_local_consensus
4932.YNL077W,YNL077W,APJ1,5,10,1.0,A_local_consensus
4932.YER103W,YER103W,SSA4,6,10,1.0,A_local_consensus
4932.YBR169C,YBR169C,SSE2,7,10,1.0,A_local_consensus
4932.YKL073W,YKL073W,LHS1,8,10,1.0,A_local_consensus
4932.YMR214W,YMR214W,SCJ1,9,10,1.0,A_local_consensus



TRACK B


,orf,preferred_name,eligible_track_rank,membership_count,membership_frequency,consensus_track
string_protein_id,,,,,,
4932.YIL030C,YIL030C,SSM4,1,2,0.2,B_cross_consensus
4932.YDL190C,YDL190C,UFD2,2,2,0.2,B_cross_consensus
4932.YBR101C,YBR101C,FES1,3,0,0.0,B_cross_consensus
4932.YGR033C,YGR033C,TIM21,4,4,0.4,Boundary
4932.YKR065C,YKR065C,PAM17,5,4,0.4,Boundary
4932.YPL098C,YPL098C,MGR2,6,4,0.4,Boundary
4932.YNL121C,YNL121C,TOM70,7,4,0.4,Boundary
4932.YJR135W-A,YJR135W-A,TIM8,8,4,0.4,Boundary
4932.YHR117W,YHR117W,TOM71,9,4,0.4,Boundary



Track A classification:
consensus_track
A_local_consensus    15
Name: count, dtype: int64

Track B classification:
consensus_track
Boundary             8
B_cross_consensus    7
Name: count, dtype: int64


In [23]:
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx

# --------------------------------------------------
# 1. Settings
# --------------------------------------------------

LOUVAIN_SEEDS = sorted(
    membership_df["seed"].unique()
)

K_VALUES = [20, 50, 100]

LAMBDA = 5.0

# --------------------------------------------------
# 2. Build weighted adjacency matrix
# --------------------------------------------------

nodes = sorted(G_hlj1.nodes())

A = nx.to_scipy_sparse_array(
    G_hlj1,
    nodelist=nodes,
    weight="confidence",
    format="csr",
    dtype=float
)

# Weighted degrees
strength = np.asarray(
    A.sum(axis=1)
).ravel()

total_volume = strength.sum()

# --------------------------------------------------
# 3. Compute community affinity for a given partition
# --------------------------------------------------

def compute_community_affinity(community_nodes):

    membership = np.array(
        [v in community_nodes for v in nodes],
        dtype=float
    )

    # Weighted links to the community
    links = np.asarray(
        A @ membership
    ).ravel()

    # Community weighted-volume prior
    prior = (
        strength[membership.astype(bool)].sum()
        / total_volume
    )

    affinity = (
        links + LAMBDA * prior
    ) / (strength + LAMBDA)

    return pd.Series(
        affinity,
        index=nodes
    )


# --------------------------------------------------
# 4. Recompute hybrid rankings for each Louvain run
# --------------------------------------------------

run_rankings = {}
run_topk = {}
run_records = []

eligible_ids = panel.index[
    panel["provisional_eligible"]
]

for seed in LOUVAIN_SEEDS:

    print(f"Evaluating Louvain seed {seed}")

    # HLJ1 community in this run
    community_s = set(
        membership_df.loc[
            (membership_df["seed"] == seed)
            & membership_df["in_hlj1_community"],
            "string_id"
        ]
    )

    assert HLJ1_ID in community_s

    # Community affinity and percentile scores
    affinity_s = compute_community_affinity(
        community_s
    )

    C_s = percentile_score(
        affinity_s.reindex(panel.index)
    )

    # Candidate scores
    run_panel = panel.loc[
        eligible_ids, ["N", "R"]
    ].copy()

    run_panel["C"] = C_s.reindex(eligible_ids)

    run_panel["track"] = np.where(
        run_panel.index.isin(community_s),
        "A_local",
        "B_cross"
    )

    run_panel["score"] = np.where(
        run_panel["track"] == "A_local",
        (
            run_panel["N"]
            + run_panel["R"]
            + run_panel["C"]
        ) / 3,
        (
            run_panel["N"]
            + run_panel["R"]
        ) / 2
    )

    # Rank within each track
    for track in ["A_local", "B_cross"]:

        ranked = (
            run_panel.loc[
                run_panel["track"] == track
            ]
            .sort_values(
                "score",
                ascending=False,
                kind="mergesort"
            )
            .copy()
        )

        ranked["rank"] = np.arange(
            1, len(ranked) + 1
        )

        run_rankings[(seed, track)] = ranked

        for k in K_VALUES:

            assert len(ranked) >= k

            run_topk[(seed, track, k)] = set(
                ranked.head(k).index
            )

        run_records.append({
            "seed": seed,
            "track": track,
            "num_candidates": len(ranked),
            "top20_count": min(20, len(ranked))
        })

print("\nEnd-to-end Louvain sensitivity completed!")

Evaluating Louvain seed 3888
Evaluating Louvain seed 3889
Evaluating Louvain seed 3890
Evaluating Louvain seed 3891
Evaluating Louvain seed 3892
Evaluating Louvain seed 3893
Evaluating Louvain seed 3894
Evaluating Louvain seed 3895
Evaluating Louvain seed 3896
Evaluating Louvain seed 3897

End-to-end Louvain sensitivity completed!


In [24]:
stability_records = []

for track in ["A_local", "B_cross"]:

    for k in K_VALUES:

        for s1, s2 in combinations(
            LOUVAIN_SEEDS, 2
        ):

            T1 = run_topk[(s1, track, k)]
            T2 = run_topk[(s2, track, k)]

            intersection = len(T1 & T2)
            union = len(T1 | T2)

            stability_records.append({
                "track": track,
                "top_k": k,
                "seed_1": s1,
                "seed_2": s2,
                "intersection": intersection,
                "jaccard": intersection / union,
            })

stability_df = pd.DataFrame(stability_records)

stability_summary = (
    stability_df
    .groupby(["track", "top_k"])
    .agg(
        mean_jaccard=("jaccard", "mean"),
        std_jaccard=("jaccard", "std"),
        min_jaccard=("jaccard", "min"),
        max_jaccard=("jaccard", "max"),
    )
    .reset_index()
)

display(stability_summary)

stability_summary.to_csv(
    OUT05 / "louvain_hybrid_stability_summary.csv",
    index=False
)

,track,top_k,mean_jaccard,std_jaccard,min_jaccard,max_jaccard
0,A_local,20,0.818081,0.080587,0.666667,1.000000
1,A_local,50,0.733626,0.126863,0.562500,0.960784
2,A_local,100,0.647108,0.139379,0.459854,0.960784
3,B_cross,20,0.405457,0.249073,0.081081,1.000000
4,B_cross,50,0.584957,0.173716,0.298701,1.000000
5,B_cross,100,0.707944,0.130177,0.481481,0.980198


In [25]:
selection_records = []

tracked_ids = (
    set(local_candidates.head(15).index)
    | set(cross_candidates.head(15).index)
)

for protein in sorted(tracked_ids):

    original_track = panel.at[protein, "track"]

    # Top-20 selection in the original track
    selection_count = sum(
        protein in run_topk[
            (seed, original_track, 20)
        ]
        for seed in LOUVAIN_SEEDS
    )

    selection_records.append({
        "string_id": protein,
        "orf": panel.at[protein, "orf"],
        "gene": panel.at[protein, "preferred_name"],
        "original_track": original_track,
        "membership_frequency":
            panel.at[protein, "membership_frequency"],
        "top20_selection_frequency":
            selection_count / len(LOUVAIN_SEEDS),
    })

selection_df = pd.DataFrame(selection_records)

print("TRACK A")
display(
    selection_df.loc[
        selection_df["original_track"] == "A_local"
    ].sort_values(
        "top20_selection_frequency",
        ascending=False
    )
)

print("\nTRACK B")
display(
    selection_df.loc[
        selection_df["original_track"] == "B_cross"
    ].sort_values(
        "top20_selection_frequency",
        ascending=False
    )
)

selection_df.to_csv(
    OUT05 / "hybrid_candidate_selection_stability.csv",
    index=False
)

TRACK A


,string_id,orf,gene,original_track,membership_frequency,top20_selection_frequency
0,4932.YBL075C,YBL075C,SSA3,A_local,1.0,1.0
2,4932.YBR169C,YBR169C,SSE2,A_local,1.0,1.0
5,4932.YDR258C,YDR258C,HSP78,A_local,1.0,1.0
6,4932.YEL030W,YEL030W,ECM10,A_local,1.0,1.0
7,4932.YER103W,YER103W,SSA4,A_local,1.0,1.0
13,4932.YJL073W,YJL073W,JEM1,A_local,1.0,1.0
14,4932.YJL162C,YJL162C,JJJ2,A_local,1.0,1.0
23,4932.YNL077W,YNL077W,APJ1,A_local,1.0,1.0
26,4932.YOR027W,YOR027W,STI1,A_local,1.0,1.0
28,4932.YPL106C,YPL106C,SSE1,A_local,1.0,1.0



TRACK B


,string_id,orf,gene,original_track,membership_frequency,top20_selection_frequency
1,4932.YBR101C,YBR101C,FES1,B_cross,0.0,1.0
10,4932.YGR184C,YGR184C,UBR1,B_cross,0.0,1.0
25,4932.YOL031C,YOL031C,SIL1,B_cross,0.0,1.0
4,4932.YDL190C,YDL190C,UFD2,B_cross,0.2,0.8
12,4932.YIL030C,YIL030C,SSM4,B_cross,0.2,0.8
20,4932.YML013W,YML013W,UBX2,B_cross,0.2,0.8
3,4932.YBR170C,YBR170C,NPL4,B_cross,0.2,0.7
8,4932.YGR033C,YGR033C,TIM21,B_cross,0.4,0.6
11,4932.YHR117W,YHR117W,TOM71,B_cross,0.4,0.6
16,4932.YJR135W-A,YJR135W-A,TIM8,B_cross,0.4,0.6


In [26]:
# --------------------------------------------------
# 1. Decompose selection frequency
# --------------------------------------------------

conditional_df = selection_df.copy()

# Probability of belonging to the original track
conditional_df["track_presence_frequency"] = np.where(
    conditional_df["original_track"] == "A_local",
    conditional_df["membership_frequency"],
    1 - conditional_df["membership_frequency"]
)

# Conditional Top-20 selection frequency
conditional_df["conditional_top20_frequency"] = (
    conditional_df["top20_selection_frequency"]
    / conditional_df["track_presence_frequency"].replace(
        0, np.nan
    )
)

# --------------------------------------------------
# 2. Validate mathematical identity
# --------------------------------------------------

reconstructed = (
    conditional_df["track_presence_frequency"]
    * conditional_df["conditional_top20_frequency"]
)

assert np.allclose(
    reconstructed,
    conditional_df["top20_selection_frequency"],
    equal_nan=False
)

assert (
    conditional_df["conditional_top20_frequency"]
    <= 1 + 1e-10
).all()

# --------------------------------------------------
# 3. Display
# --------------------------------------------------

columns = [
    "gene",
    "original_track",
    "track_presence_frequency",
    "top20_selection_frequency",
    "conditional_top20_frequency",
]

print("Conditional Ranking Stability")

display(
    conditional_df[columns].sort_values(
        ["original_track", "conditional_top20_frequency"],
        ascending=[True, False]
    )
)

# Save
conditional_df.to_csv(
    OUT05 / "conditional_candidate_stability.csv",
    index=False
)

Conditional Ranking Stability


,gene,original_track,track_presence_frequency,top20_selection_frequency,conditional_top20_frequency
0,SSA3,A_local,1.0,1.0,1.000
2,SSE2,A_local,1.0,1.0,1.000
5,HSP78,A_local,1.0,1.0,1.000
6,ECM10,A_local,1.0,1.0,1.000
7,SSA4,A_local,1.0,1.0,1.000
13,JEM1,A_local,1.0,1.0,1.000
14,JJJ2,A_local,1.0,1.0,1.000
23,APJ1,A_local,1.0,1.0,1.000
26,STI1,A_local,1.0,1.0,1.000
28,SSE1,A_local,1.0,1.0,1.000


In [27]:
# --------------------------------------------------
# 1. Independent validation seeds
# --------------------------------------------------

VALIDATION_SEEDS = list(range(3898, 3908))

validation_clusters = {}

for seed in VALIDATION_SEEDS:

    print(f"Validation Louvain seed={seed}")

    communities = nx.community.louvain_communities(
        G,
        weight="confidence",
        resolution=1.0,
        seed=seed
    )

    hlj1_cluster = next(
        c for c in communities
        if HLJ1_ID in c
    )

    validation_clusters[seed] = set(hlj1_cluster)

# --------------------------------------------------
# 2. Compute held-out membership frequencies
# --------------------------------------------------

validation_counts = pd.Series(
    {
        v: sum(
            v in cluster
            for cluster in validation_clusters.values()
        )
        for v in panel.index
    },
    name="validation_membership_count"
)

panel["validation_membership_count"] = (
    validation_counts
)

panel["validation_membership_frequency"] = (
    panel["validation_membership_count"]
    / len(VALIDATION_SEEDS)
)

# --------------------------------------------------
# 3. Held-out consensus classifications
# --------------------------------------------------

panel["validation_consensus_track"] = np.select(
    [
        panel["validation_membership_count"] >= 8,
        panel["validation_membership_count"] <= 2,
    ],
    [
        "A_local_consensus",
        "B_cross_consensus",
    ],
    default="Boundary"
)

# --------------------------------------------------
# 4. Compare training and validation
# --------------------------------------------------

print("Training vs Validation Consensus")

display(
    pd.crosstab(
        panel["consensus_track"],
        panel["validation_consensus_track"]
    )
)

# --------------------------------------------------
# 5. Inspect original Top-15 candidates
# --------------------------------------------------

tracked_ids = (
    set(local_candidates.head(15).index)
    | set(cross_candidates.head(15).index)
)

validation_table = panel.loc[
    sorted(tracked_ids),
    [
        "orf",
        "preferred_name",
        "track",
        "membership_frequency",
        "validation_membership_frequency",
        "consensus_track",
        "validation_consensus_track"
    ]
].copy()

validation_table["membership_change"] = (
    validation_table["validation_membership_frequency"]
    - validation_table["membership_frequency"]
)

display(
    validation_table.sort_values(
        "membership_change"
    )
)

validation_table.to_csv(
    OUT05 / "consensus_heldout_validation.csv"
)

Validation Louvain seed=3898
Validation Louvain seed=3899
Validation Louvain seed=3900
Validation Louvain seed=3901
Validation Louvain seed=3902
Validation Louvain seed=3903
Validation Louvain seed=3904
Validation Louvain seed=3905
Validation Louvain seed=3906
Validation Louvain seed=3907
Training vs Validation Consensus


validation_consensus_track,A_local_consensus,B_cross_consensus,Boundary
consensus_track,,,
A_local_consensus,123,0,44
B_cross_consensus,0,4889,250
Boundary,3,34,372


,orf,preferred_name,track,membership_frequency,validation_membership_frequency,consensus_track,validation_consensus_track,membership_change
string_protein_id,,,,,,,,
4932.YBL075C,YBL075C,SSA3,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YOR027W,YOR027W,STI1,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YOL031C,YOL031C,SIL1,B_cross,0.0,0.0,B_cross_consensus,B_cross_consensus,0.0
4932.YNL077W,YNL077W,APJ1,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YMR214W,YMR214W,SCJ1,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YML013W,YML013W,UBX2,B_cross,0.2,0.2,B_cross_consensus,B_cross_consensus,0.0
4932.YLR369W,YLR369W,SSQ1,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YKL073W,YKL073W,LHS1,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0
4932.YJR097W,YJR097W,JJJ3,A_local,1.0,1.0,A_local_consensus,A_local_consensus,0.0


In [28]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Reconstruct training Louvain communities
# --------------------------------------------------

train_clusters = {
    int(seed): set(
        group.loc[
            group["in_hlj1_community"],
            "string_id"
        ]
    )
    for seed, group in membership_df.groupby("seed")
}

# Validation clusters already computed in Cell 23
test_clusters = validation_clusters

assert len(train_clusters) == 10
assert len(test_clusters) == 10

assert set(train_clusters).isdisjoint(set(test_clusters))

assert all(
    HLJ1_ID in cluster
    for cluster in [
        *train_clusters.values(),
        *test_clusters.values()
    ]
)


# --------------------------------------------------
# 2. Fit one consensus-aware model
# --------------------------------------------------

def fit_consensus_hybrid(clusters, panel, threshold=0.8):

    n_runs = len(clusters)

    # Community membership counts
    counts = pd.Series(
        0,
        index=panel.index,
        dtype=int
    )

    # Accumulate raw community affinity
    affinity_sum = pd.Series(
        0.0,
        index=panel.index
    )

    for cluster in clusters.values():

        counts += pd.Series(
            panel.index.isin(cluster).astype(int),
            index=panel.index
        )

        affinity_s = compute_community_affinity(
            cluster
        ).reindex(panel.index)

        affinity_sum += affinity_s

    # --------------------------------------------------
    # 3. Consensus quantities
    # --------------------------------------------------

    membership = counts / n_runs

    mean_affinity = affinity_sum / n_runs

    C_consensus = percentile_score(mean_affinity)

    # Integer thresholds
    min_local_votes = int(
        np.ceil(threshold * n_runs - 1e-12)
    )

    max_cross_votes = n_runs - min_local_votes

    tracks = np.select(
        [
            counts >= min_local_votes,
            counts <= max_cross_votes
        ],
        [
            "A_local_consensus",
            "B_cross_consensus"
        ],
        default="Boundary"
    )

    # --------------------------------------------------
    # 4. Construct scoring panel
    # --------------------------------------------------

    result = panel[
        [
            "orf",
            "preferred_name",
            "N",
            "R",
            "provisional_eligible"
        ]
    ].copy()

    result["membership_frequency"] = membership

    result["consensus_track"] = tracks

    result["C_consensus"] = C_consensus

    # Local: N + R + C
    # Cross / Boundary: N + R

    local_score = (
        result["N"]
        + result["R"]
        + result["C_consensus"]
    ) / 3

    other_score = (
        result["N"]
        + result["R"]
    ) / 2

    result["hybrid_score"] = np.where(
        result["consensus_track"] == "A_local_consensus",
        local_score,
        other_score
    )

    # --------------------------------------------------
    # 5. Rank eligible proteins within each track
    # --------------------------------------------------

    eligible_result = result.loc[
        result["provisional_eligible"]
    ].copy()

    eligible_result["string_id"] = (
        eligible_result.index
    )

    rankings = {}

    for track in [
        "A_local_consensus",
        "B_cross_consensus",
        "Boundary"
    ]:

        ranked = (
            eligible_result.loc[
                eligible_result["consensus_track"] == track
            ]
            .sort_values(
                ["hybrid_score", "string_id"],
                ascending=[False, True],
                kind="mergesort"
            )
            .copy()
        )

        ranked["rank"] = np.arange(
            1, len(ranked) + 1
        )

        rankings[track] = ranked

    return result, rankings


# --------------------------------------------------
# 6. Fit independent consensus models
# --------------------------------------------------

train_model, train_rankings = fit_consensus_hybrid(
    train_clusters,
    panel
)

test_model, test_rankings = fit_consensus_hybrid(
    test_clusters,
    panel
)

print("Both consensus-aware models fitted successfully!")

Both consensus-aware models fitted successfully!


In [29]:
TRACKS = [
    "A_local_consensus",
    "B_cross_consensus",
    "Boundary"
]

K_VALUES = [20, 50, 100]

comparison_records = []

for track in TRACKS:

    train_ranking = train_rankings[track]
    test_ranking = test_rankings[track]

    for k in K_VALUES:

        A = set(train_ranking.head(k).index)
        B = set(test_ranking.head(k).index)

        intersection = len(A & B)
        union = len(A | B)

        comparison_records.append({
            "track": track,
            "top_k": k,
            "train_pool_size": len(train_ranking),
            "test_pool_size": len(test_ranking),
            "intersection": intersection,
            "jaccard": (
                intersection / union
                if union > 0 else np.nan
            )
        })

consensus_comparison = pd.DataFrame(
    comparison_records
)

display(consensus_comparison)

consensus_comparison.to_csv(
    OUT05 / "consensus_ranking_validation.csv",
    index=False
)

,track,top_k,train_pool_size,test_pool_size,intersection,jaccard
0,A_local_consensus,20,120,97,19,0.904762
1,A_local_consensus,50,120,97,45,0.818182
2,A_local_consensus,100,120,97,78,0.655462
3,B_cross_consensus,20,3985,3807,20,1.000000
4,B_cross_consensus,50,3985,3807,48,0.923077
5,B_cross_consensus,100,3985,3807,98,0.960784
6,Boundary,20,334,535,15,0.600000
7,Boundary,50,334,535,32,0.470588
8,Boundary,100,334,535,65,0.481481


In [30]:
# --------------------------------------------------
# 1. Original candidate proteins
# --------------------------------------------------

tracked_ids = (
    set(local_candidates.head(15).index)
    | set(cross_candidates.head(15).index)
)

# --------------------------------------------------
# 2. Build comparison table
# --------------------------------------------------

records = []

for protein in sorted(tracked_ids):

    train_track = train_model.at[
        protein, "consensus_track"
    ]

    test_track = test_model.at[
        protein, "consensus_track"
    ]

    # Retrieve ranks
    train_rank = (
        train_rankings[train_track]["rank"].get(
            protein, np.nan
        )
    )

    test_rank = (
        test_rankings[test_track]["rank"].get(
            protein, np.nan
        )
    )

    records.append({
        "orf": panel.at[protein, "orf"],
        "gene": panel.at[protein, "preferred_name"],
        "train_track": train_track,
        "test_track": test_track,
        "train_rank": train_rank,
        "test_rank": test_rank,
        "train_membership":
            train_model.at[
                protein, "membership_frequency"
            ],
        "test_membership":
            test_model.at[
                protein, "membership_frequency"
            ],
        "same_track": train_track == test_track
    })

candidate_validation = pd.DataFrame(records)

# Rank differences are meaningful only
# when both ensembles assign the same track.

candidate_validation["rank_change"] = np.where(
    candidate_validation["same_track"],
    candidate_validation["test_rank"]
    - candidate_validation["train_rank"],
    np.nan
)

display(
    candidate_validation.sort_values(
        ["same_track", "train_track", "train_rank"],
        ascending=[False, True, True]
    )
)

candidate_validation.to_csv(
    OUT05 / "consensus_candidate_validation.csv",
    index=False
)

,orf,gene,train_track,test_track,train_rank,test_rank,train_membership,test_membership,same_track,rank_change
28,YPL106C,SSE1,A_local_consensus,A_local_consensus,1,1,1.0,1.0,True,0.0
23,YNL077W,APJ1,A_local_consensus,A_local_consensus,2,2,1.0,1.0,True,0.0
14,YJL162C,JJJ2,A_local_consensus,A_local_consensus,3,3,1.0,1.0,True,0.0
29,YPR061C,JID1,A_local_consensus,A_local_consensus,5,5,1.0,1.0,True,0.0
7,YER103W,SSA4,A_local_consensus,A_local_consensus,6,7,1.0,1.0,True,1.0
0,YBL075C,SSA3,A_local_consensus,A_local_consensus,7,9,1.0,1.0,True,2.0
6,YEL030W,ECM10,A_local_consensus,A_local_consensus,8,6,1.0,1.0,True,-2.0
2,YBR169C,SSE2,A_local_consensus,A_local_consensus,11,10,1.0,1.0,True,-1.0
17,YKL073W,LHS1,A_local_consensus,A_local_consensus,12,13,1.0,1.0,True,1.0
15,YJR097W,JJJ3,A_local_consensus,A_local_consensus,14,15,1.0,1.0,True,1.0


In [31]:
# --------------------------------------------------
# 1. Same eligible candidate universe
# --------------------------------------------------

eligible_ids = panel.index[panel["provisional_eligible"]]

train_labels = train_model.loc[eligible_ids, "consensus_track"]

test_labels = test_model.loc[eligible_ids, "consensus_track"]

# --------------------------------------------------
# 2. Classification agreement
# --------------------------------------------------

agreement = (train_labels == test_labels)

print(f"Overall agreement: {agreement.mean():.2%}")

print(f"Changed classification: {(~agreement).sum():,}")

# --------------------------------------------------
# 3. Transition matrix
# --------------------------------------------------

transition_matrix = pd.crosstab(
    train_labels.rename("Training"),
    test_labels.rename("Validation")
)

display(transition_matrix)

# --------------------------------------------------
# 4. Per-track retention
# --------------------------------------------------

track_retention = []

for track in [
    "A_local_consensus",
    "B_cross_consensus",
    "Boundary"
]:

    train_set = set(train_labels.index[train_labels == track])

    test_set = set(test_labels.index[test_labels == track])

    intersection = len(train_set & test_set)
    union = len(train_set | test_set)

    track_retention.append({
        "track": track,
        "train_size": len(train_set),
        "test_size": len(test_set),
        "intersection": intersection,
        "jaccard": intersection / union
    })

display(pd.DataFrame(track_retention))

Overall agreement: 93.85%
Changed classification: 273


Validation,A_local_consensus,B_cross_consensus,Boundary
Training,,,
A_local_consensus,95,0,25
B_cross_consensus,0,3773,212
Boundary,2,34,298


,track,train_size,test_size,intersection,jaccard
0,A_local_consensus,120,97,95,0.778689
1,B_cross_consensus,3985,3807,3773,0.938791
2,Boundary,334,535,298,0.521891


In [32]:
# --------------------------------------------------
# 1. Build final candidate evidence table
# --------------------------------------------------

TRACKS = [
    "A_local_consensus",
    "B_cross_consensus",
    "Boundary"
]

records = []

for track in TRACKS:

    ranked = train_rankings[track]

    for protein, row in ranked.head(20).iterrows():

        # Rank under held-out ensemble
        validation_rank = (
            test_rankings[track]["rank"].get(
                protein, np.nan
            )
        )

        records.append({
            "string_id": protein,
            "orf": row["orf"],
            "gene": row["preferred_name"],
            "track": track,
            "train_rank": row["rank"],
            "validation_rank": validation_rank,
            "validation_track":
                test_model.at[
                    protein, "consensus_track"
                ],
            "hybrid_score": row["hybrid_score"],
            "node2vec_score": row["N"],
            "rwr_score": row["R"],
            "community_score": row["C_consensus"],
            "train_membership":
                train_model.at[
                    protein, "membership_frequency"
                ],
            "validation_membership":
                test_model.at[
                    protein, "membership_frequency"
                ],
        })

final_candidates = pd.DataFrame(records)

# --------------------------------------------------
# 2. Add network and experimental information
# --------------------------------------------------

feature_cols = [
    "degree",
    "h_hops",
    "community_id",
    "physical_degree_700",
    "direct_S_neighbours",
    "jaccard_S_mean",
    "is_listed_essential",
]

available_cols = [
    col for col in feature_cols
    if col in panel.columns
]

final_candidates = final_candidates.join(
    panel[available_cols],
    on="string_id",
    validate="many_to_one"
)

# --------------------------------------------------
# 3. Validation diagnostics
# --------------------------------------------------

final_candidates["track_consistent"] = (
    final_candidates["track"]
    == final_candidates["validation_track"]
)

final_candidates["rank_change"] = np.where(
    final_candidates["track_consistent"],
    final_candidates["validation_rank"]
    - final_candidates["train_rank"],
    np.nan
)

# --------------------------------------------------
# 4. Export
# --------------------------------------------------

final_candidates.to_csv(
    OUT05 / "final_cadtp_candidate_evidence.csv",
    index=False
)

for track in TRACKS:

    print(f"\n{track}")

    display(
        final_candidates.loc[
            final_candidates["track"] == track
        ].head(10)
    )

print("\nFinal CADTP candidate evidence exported!")


A_local_consensus


,string_id,orf,gene,track,train_rank,validation_rank,validation_track,hybrid_score,node2vec_score,rwr_score,...,validation_membership,degree,h_hops,community_id,physical_degree_700,direct_S_neighbours,jaccard_S_mean,is_listed_essential,track_consistent,rank_change
0,4932.YPL106C,YPL106C,SSE1,A_local_consensus,1,1.0,A_local_consensus,0.999096,0.998687,0.998775,...,1.0,51,1.0,4,6,0,0.012893,False,True,0.0
1,4932.YNL077W,YNL077W,APJ1,A_local_consensus,2,2.0,A_local_consensus,0.998075,0.999825,0.995275,...,1.0,38,2.0,4,5,0,0.012694,False,True,0.0
2,4932.YJL162C,YJL162C,JJJ2,A_local_consensus,3,3.0,A_local_consensus,0.998017,1.000000,0.994575,...,1.0,31,2.0,4,0,0,0.014294,False,True,0.0
3,4932.YFR041C,YFR041C,ERJ5,A_local_consensus,4,4.0,A_local_consensus,0.997958,0.999475,0.997725,...,1.0,22,1.0,4,0,0,0.017066,False,True,0.0
4,4932.YPR061C,YPR061C,JID1,A_local_consensus,5,5.0,A_local_consensus,0.997871,0.998162,0.997900,...,1.0,26,1.0,4,0,0,0.000000,False,True,0.0
5,4932.YER103W,YER103W,SSA4,A_local_consensus,6,7.0,A_local_consensus,0.997754,0.998162,0.997200,...,1.0,36,1.0,4,6,0,0.006207,False,True,1.0
6,4932.YBL075C,YBL075C,SSA3,A_local_consensus,7,9.0,A_local_consensus,0.997696,0.997287,0.997375,...,1.0,32,1.0,4,6,0,0.000000,False,True,2.0
7,4932.YEL030W,YEL030W,ECM10,A_local_consensus,8,6.0,A_local_consensus,0.997696,0.998162,0.996850,...,1.0,29,1.0,4,2,0,0.000000,False,True,-2.0
8,4932.YFL016C,YFL016C,MDJ1,A_local_consensus,9,8.0,A_local_consensus,0.997550,0.998950,0.995100,...,1.0,40,2.0,4,0,0,0.012301,False,True,-1.0
9,4932.YAL005C,YAL005C,SSA1,A_local_consensus,10,12.0,A_local_consensus,0.997229,0.997637,0.998600,...,1.0,61,1.0,4,12,0,0.028127,False,True,2.0



B_cross_consensus


,string_id,orf,gene,track,train_rank,validation_rank,validation_track,hybrid_score,node2vec_score,rwr_score,...,validation_membership,degree,h_hops,community_id,physical_degree_700,direct_S_neighbours,jaccard_S_mean,is_listed_essential,track_consistent,rank_change
20,4932.YIL030C,YIL030C,SSM4,B_cross_consensus,1,1.0,B_cross_consensus,0.991162,0.986699,0.995625,...,0.2,34,1.0,9,13,5,0.380422,False,True,0.0
21,4932.YDL190C,YDL190C,UFD2,B_cross_consensus,2,2.0,B_cross_consensus,0.990200,0.984949,0.995450,...,0.2,28,1.0,9,8,0,0.128691,False,True,0.0
22,4932.YBR101C,YBR101C,FES1,B_cross_consensus,3,3.0,B_cross_consensus,0.988712,0.984074,0.993350,...,0.0,238,2.0,5,8,0,0.006864,False,True,0.0
23,4932.YOL031C,YOL031C,SIL1,B_cross_consensus,4,4.0,B_cross_consensus,0.979261,0.971649,0.986874,...,0.0,218,2.0,5,5,0,0.010773,False,True,0.0
24,4932.YML013W,YML013W,UBX2,B_cross_consensus,5,5.0,B_cross_consensus,0.979130,0.977686,0.980574,...,0.2,31,2.0,9,17,5,0.323339,False,True,0.0
25,4932.YGR184C,YGR184C,UBR1,B_cross_consensus,6,6.0,B_cross_consensus,0.978605,0.979786,0.977424,...,0.0,44,2.0,9,26,1,0.053836,False,True,0.0
26,4932.YBR170C,YBR170C,NPL4,B_cross_consensus,7,7.0,B_cross_consensus,0.973967,0.962986,0.984949,...,0.2,73,2.0,9,51,5,0.137807,False,True,0.0
27,4932.YLR143W,YLR143W,DPH6,B_cross_consensus,8,8.0,B_cross_consensus,0.971474,0.985824,0.957123,...,0.1,8,2.0,4,0,0,0.000000,False,True,0.0
28,4932.YDR057W,YDR057W,YOS9,B_cross_consensus,9,9.0,B_cross_consensus,0.971036,0.978649,0.963423,...,0.2,21,2.0,9,13,5,0.461286,False,True,0.0
29,4932.YDR411C,YDR411C,DFM1,B_cross_consensus,10,10.0,B_cross_consensus,0.970599,0.981624,0.959573,...,0.2,19,2.0,9,13,5,0.412676,False,True,0.0



Boundary


,string_id,orf,gene,track,train_rank,validation_rank,validation_track,hybrid_score,node2vec_score,rwr_score,...,validation_membership,degree,h_hops,community_id,physical_degree_700,direct_S_neighbours,jaccard_S_mean,is_listed_essential,track_consistent,rank_change
40,4932.YGR033C,YGR033C,TIM21,Boundary,1,1.0,Boundary,0.987224,0.986699,0.987749,...,0.6,33,2.0,7,12,0,0.000000,False,True,0.0
41,4932.YPL135W,YPL135W,ISU1,Boundary,2,2.0,Boundary,0.987137,0.987749,0.986524,...,0.7,22,2.0,4,5,0,0.000000,False,True,0.0
42,4932.YKR065C,YKR065C,PAM17,Boundary,3,3.0,Boundary,0.986043,0.988012,0.984074,...,0.6,17,2.0,7,11,0,0.000000,False,True,0.0
43,4932.YPL098C,YPL098C,MGR2,Boundary,4,5.0,Boundary,0.985256,0.988012,0.982499,...,0.6,20,2.0,7,11,0,0.000000,False,True,1.0
44,4932.YNL121C,YNL121C,TOM70,Boundary,5,6.0,Boundary,0.984599,0.979174,0.990025,...,0.6,59,2.0,7,21,0,0.024185,False,True,1.0
45,4932.YKL040C,YKL040C,NFU1,Boundary,6,7.0,Boundary,0.983112,0.983549,0.982674,...,0.7,29,2.0,4,0,0,0.000000,False,True,1.0
46,4932.YPR067W,YPR067W,ISA2,Boundary,7,8.0,Boundary,0.983112,0.986349,0.979874,...,0.7,19,2.0,4,2,0,0.000000,False,True,1.0
47,4932.YFL014W,YFL014W,HSP12,Boundary,8,9.0,Boundary,0.981274,0.978824,0.983724,...,0.6,27,2.0,4,0,0,0.000000,False,True,1.0
48,4932.YLL027W,YLL027W,ISA1,Boundary,9,10.0,Boundary,0.981187,0.981974,0.980399,...,0.7,25,2.0,4,2,0,0.000000,False,True,1.0
49,4932.YJR135W-A,YJR135W-A,TIM8,Boundary,10,11.0,Boundary,0.980399,0.983024,0.977774,...,0.6,32,2.0,7,4,0,0.000000,False,True,1.0



Final CADTP candidate evidence exported!


In [33]:
# Export the complete eligible candidate universe
# Run this cell in Notebook 05

score_columns = [
    "orf",
    "preferred_name",
    "N",
    "R",
    "C_consensus",
    "hybrid_score",
    "consensus_track",
    "membership_frequency",
    "provisional_eligible",
]

full_scores = train_model.loc[
    train_model["provisional_eligible"],
    score_columns
].copy()

# Add topological features
full_scores = full_scores.join(
    panel[["degree", "h_hops"]],
    validate="one_to_one"
)

full_scores.index.name = "string_id"

assert len(full_scores) == 4439
assert full_scores["orf"].notna().all()

full_scores.to_csv(
    OUT05 / "cadtp_full_candidate_scores.csv.gz",
    compression="gzip"
)

print(f"Exported {len(full_scores):,} eligible proteins")

Exported 4,439 eligible proteins
